# 02 · Preprocessing
Missing-value analysis, duplicate analysis, and data-type correction for `application_train.csv`.
No rows/columns are silently dropped without a documented reason.

In [1]:
import pandas as pd
import numpy as np

app_train = pd.read_csv('../data/application_train.csv')
app_train.shape


(307511, 122)

## Step 2 — Missing Values

In [2]:
missing = app_train.isnull().sum()
missing_pct = (missing / len(app_train) * 100).round(2)
missing_table = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_table = missing_table[missing_table['missing_count'] > 0].sort_values('missing_pct', ascending=False)
missing_table.head(20)


,missing_count,missing_pct
COMMONAREA_MEDI,214865,69.87
COMMONAREA_MODE,214865,69.87
COMMONAREA_AVG,214865,69.87
NONLIVINGAPARTMENTS_MODE,213514,69.43
NONLIVINGAPARTMENTS_MEDI,213514,69.43
NONLIVINGAPARTMENTS_AVG,213514,69.43
FONDKAPREMONT_MODE,210295,68.39
LIVINGAPARTMENTS_AVG,210199,68.35
LIVINGAPARTMENTS_MEDI,210199,68.35
LIVINGAPARTMENTS_MODE,210199,68.35


In [3]:
def bucket(pct):
    if pct <= 5: return '0-5%'
    if pct <= 20: return '5-20%'
    if pct <= 40: return '20-40%'
    if pct <= 60: return '40-60%'
    return '60%+'

missing_table['bucket'] = missing_table['missing_pct'].apply(bucket)
missing_table['bucket'].value_counts()


bucket
40-60%    32
60%+      17
0-5%      10
5-20%      7
20-40%     1
Name: count, dtype: int64

## Step 3 — Duplicate Analysis

In [4]:
print('Full row duplicates:', app_train.duplicated().sum())
print('Duplicate SK_ID_CURR:', app_train['SK_ID_CURR'].duplicated().sum())
print('Unique customers:', app_train['SK_ID_CURR'].nunique())


Full row duplicates: 0
Duplicate SK_ID_CURR: 0
Unique customers: 307511


## Step 4 — Data Type Correction

In [5]:
# DAYS_* columns are stored as negative day counts relative to the application date.
# Convert to positive, human-readable years instead of using them raw.
app_train['AGE_YEARS'] = (-app_train['DAYS_BIRTH'] / 365.25).round(1)

# DAYS_EMPLOYED uses a known sentinel value (365243) for pensioners / not employed.
app_train['DAYS_EMPLOYED_CLEAN'] = app_train['DAYS_EMPLOYED'].replace(365243, np.nan)
app_train['EMPLOYMENT_YEARS'] = (-app_train['DAYS_EMPLOYED_CLEAN'] / 365.25).round(1)

app_train[['DAYS_BIRTH', 'AGE_YEARS', 'DAYS_EMPLOYED', 'EMPLOYMENT_YEARS']].head()


/tmp/ipykernel_927/1633268697.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app_train['AGE_YEARS'] = (-app_train['DAYS_BIRTH'] / 365.25).round(1)
/tmp/ipykernel_927/1633268697.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app_train['DAYS_EMPLOYED_CLEAN'] = app_train['DAYS_EMPLOYED'].replace(365243, np.nan)
/tmp/ipykernel_927/1633268697.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at

,DAYS_BIRTH,AGE_YEARS,DAYS_EMPLOYED,EMPLOYMENT_YEARS
0,-9461,25.9,-637,1.7
1,-16765,45.9,-1188,3.3
2,-19046,52.1,-225,0.6
3,-19005,52.0,-3039,8.3
4,-19932,54.6,-3038,8.3


## Step 5 — Invalid Value Investigation

In [6]:
print('DAYS_EMPLOYED sentinel count (365243):', (app_train['DAYS_EMPLOYED'] == 365243).sum())
print('CNT_CHILDREN max:', app_train['CNT_CHILDREN'].max())
print('CNT_FAM_MEMBERS missing:', app_train['CNT_FAM_MEMBERS'].isnull().sum())


DAYS_EMPLOYED sentinel count (365243): 55374
CNT_CHILDREN max: 19
CNT_FAM_MEMBERS missing: 2


## Step 6 — Outlier Analysis (IQR)

In [7]:
def iqr_outliers(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((s < lower) | (s > upper)).sum()

for col in ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE']:
    print(col, '->', iqr_outliers(app_train[col].dropna()), 'flagged (not removed)')


AMT_INCOME_TOTAL -> 14035 flagged (not removed)
AMT_CREDIT -> 6562 flagged (not removed)
AMT_ANNUITY -> 7504 flagged (not removed)
AMT_GOODS_PRICE -> 14728 flagged (not removed)


## Preprocessing Decisions Summary

| Column Group | Decision | Reason |
|---|---|---|
| >60% missing (e.g. `COMMONAREA_*`) | Drop or keep as missing-indicator only | Too sparse to impute reliably |
| `OCCUPATION_TYPE`, `EXT_SOURCE_1` (20-40%) | Mode/median fill, or explicit "Unknown" category | Business-meaningful gap |
| `AMT_ANNUITY`, `AMT_GOODS_PRICE`, `CNT_FAM_MEMBERS` (<1%) | Median/mode fill | Negligible impact |
| `DAYS_EMPLOYED` sentinel (365243) | Recode to NaN + "Unemployed/Special" category | Known data convention, not a true outlier |
| `DAYS_BIRTH`, `DAYS_EMPLOYED`, etc. | Convert to positive year features | Original encoding is not analysis-friendly |
